# LAB1 - Bài 13 (Online Retail): FP-Growth và so sánh với Apriori (bước 12)

**Họ tên:** Đinh Công Vinh Quang  
**MSSV:** 6451071061

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

HO_TEN, MSSV = "Đinh Công Vinh Quang", "6451071061"
print(f"{HO_TEN} - {MSSV}")
import time
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

Đinh Công Vinh Quang - 6451071061


In [2]:
# 1. Đọc dữ liệu
df = pd.read_excel("Online_Retail.xlsx")
print("Kích thước ban đầu:", df.shape)

# 2. Cắt ký tự thừa ở Description, xóa dòng không có InvoiceNo, đổi InvoiceNo về chuỗi
df["Description"] = df["Description"].str.strip()
df = df.dropna(axis=0, subset=["InvoiceNo"])
df["InvoiceNo"] = df["InvoiceNo"].astype("str")

# 3. Xóa hóa đơn tín dụng (InvoiceNo chứa 'C')
credit = df[df["InvoiceNo"].str.contains("C")]
df = df[~df["InvoiceNo"].str.contains("C")]
print("Số dòng hóa đơn tín dụng đã xóa:", len(credit), "| còn lại:", len(df))
# 5. Hóa đơn nước Anh, gom nhóm Quantity theo InvoiceNo và Description
def make_basket(data, country):
    return (data[data["Country"] == country]
            .groupby(["InvoiceNo", "Description"])["Quantity"].sum().unstack(fill_value=0))

# 7. Chuyển về hot/one-hot encoding: Quantity > 0 -> 1 (True), còn lại 0 (False)
def encode_units(x):
    return x > 0

basket = make_basket(df, "United Kingdom")                 # 6. mỗi dòng là một hóa đơn
basket_sets = encode_units(basket)
# 8. Xóa cột POSTAGE (cước phí)
if "POSTAGE" in basket_sets.columns:
    basket_sets = basket_sets.drop(columns=["POSTAGE"])
print("Ma trận hóa đơn x mặt hàng:", basket_sets.shape)

Kích thước ban đầu: (541909, 8)


Số dòng hóa đơn tín dụng đã xóa: 9288 | còn lại: 532621


Ma trận hóa đơn x mặt hàng: (18667, 4174)


In [3]:
def compare(basket_sets, min_sup, min_conf):
    t = time.perf_counter(); ap = apriori(basket_sets, min_support=min_sup, use_colnames=True); t_ap = time.perf_counter() - t
    t = time.perf_counter(); fp = fpgrowth(basket_sets, min_support=min_sup, use_colnames=True); t_fp = time.perf_counter() - t
    same = ({(frozenset(a), round(s, 6)) for a, s in zip(ap["itemsets"], ap["support"])} ==
            {(frozenset(a), round(s, 6)) for a, s in zip(fp["itemsets"], fp["support"])})
    r_ap = association_rules(ap, metric="confidence", min_threshold=min_conf) if len(ap) else ap
    r_fp = association_rules(fp, metric="confidence", min_threshold=min_conf) if len(fp) else fp
    print(f"Apriori  : {len(ap)} tập phổ biến, {len(r_ap)} luật, {t_ap:.2f}s")
    print(f"FP-Growth: {len(fp)} tập phổ biến, {len(r_fp)} luật, {t_fp:.2f}s")
    print("Hai thuật toán cho cùng tập phổ biến và support:", same)
    return ap, r_ap
ap, r_ap = compare(basket_sets, 0.03, 0.5)

Apriori  : 131 tập phổ biến, 10 luật, 1.18s
FP-Growth: 131 tập phổ biến, 10 luật, 0.43s
Hai thuật toán cho cùng tập phổ biến và support: True


In [4]:
# tập phổ biến và luật tìm bằng FP-Growth (min_sup = 3%, min_conf = 50%)
fp = fpgrowth(basket_sets, min_support=0.03, use_colnames=True)
rules_fp = association_rules(fp, metric="confidence", min_threshold=0.5)
rules_fp["antecedents"] = rules_fp["antecedents"].apply(lambda s: ", ".join(sorted(s)))
rules_fp["consequents"] = rules_fp["consequents"].apply(lambda s: ", ".join(sorted(s)))
for _, r in rules_fp.iterrows():
    print(f"{r['antecedents']}  =>  {r['consequents']}   (support={r['support']:.3f}, confidence={r['confidence']:.3f})")

JUMBO BAG PINK POLKADOT  =>  JUMBO BAG RED RETROSPOT   (support=0.042, confidence=0.677)
JUMBO STORAGE BAG SUKI  =>  JUMBO BAG RED RETROSPOT   (support=0.037, confidence=0.618)
JUMBO  BAG BAROQUE BLACK WHITE  =>  JUMBO BAG RED RETROSPOT   (support=0.031, confidence=0.626)
JUMBO SHOPPER VINTAGE RED PAISLEY  =>  JUMBO BAG RED RETROSPOT   (support=0.035, confidence=0.580)
ALARM CLOCK BAKELIKE GREEN  =>  ALARM CLOCK BAKELIKE RED   (support=0.030, confidence=0.643)
ALARM CLOCK BAKELIKE RED  =>  ALARM CLOCK BAKELIKE GREEN   (support=0.030, confidence=0.605)
GREEN REGENCY TEACUP AND SAUCER  =>  ROSES REGENCY TEACUP AND SAUCER   (support=0.038, confidence=0.751)
ROSES REGENCY TEACUP AND SAUCER  =>  GREEN REGENCY TEACUP AND SAUCER   (support=0.038, confidence=0.732)
GREEN REGENCY TEACUP AND SAUCER  =>  PINK REGENCY TEACUP AND SAUCER   (support=0.031, confidence=0.618)
PINK REGENCY TEACUP AND SAUCER  =>  GREEN REGENCY TEACUP AND SAUCER   (support=0.031, confidence=0.821)


**Nhận xét:** FP-Growth và Apriori cho **cùng** tập phổ biến và cùng tập luật (kết quả khác nhau chỉ ở thứ tự); khác biệt chính là thời gian chạy (xem số liệu ở trên): FP-Growth không sinh tập ứng viên nên thường nhanh hơn, càng rõ khi min_sup nhỏ.